# MSNet Behavioral Fusion for EMS Schizophrenia Recognition

This notebook reproduces the official pretrained MSNet validation baseline, builds behavioral eye-movement features, evaluates conventional machine-learning baselines, performs probability-level MSNet + behavioral fusion, generates XAI/UMAP outputs, and prepares the additional analyses requested by the instructor.

Added final outputs include:

- comparison with Logistic Regression, Random Forest, HistGradientBoosting and the official EMS benchmark;
- validation confusion matrices for reproduced MSNet and the proposed fusion;
- subject-level examples corrected by fusion and remaining failure cases;
- a clear separation between 4-fold validation results and the held-out EMS test set;
- generation of the final `prob_test.txt` for the proposed fusion model, ready to email to the EMS authors;
- a submission metadata file and email template.

Main package output: `outputs/improve_behavioral_final_results.zip`.

## 1. Environment and input discovery

Inspect the Kaggle environment and locate the EMS dataset, pretrained MSNet files, and RINet feature dictionary.

In [ ]:
import os
import sys
import glob
import torch
from pathlib import Path


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "EMS").is_dir() and (candidate / "EMS_repo").is_dir():
            return candidate
    raise FileNotFoundError(
        "Cannot locate the project root containing EMS and EMS_repo. "
        "Start Jupyter from the repository root."
    )


PROJECT_ROOT = find_project_root()
WORK_DIR = str(PROJECT_ROOT)
REPO_DIR = str(PROJECT_ROOT / "EMS_repo")
DATASET_DIR = str(PROJECT_ROOT / "EMS")
FEATURE_DICT = str(PROJECT_ROOT / "feature_dict_RINet.npy")
MODEL_SRC = str(PROJECT_ROOT / "model_parameters")
OUT_DIR = str(PROJECT_ROOT / "outputs" / "improve_behavioral")
BASELINE_BACKUP_DIR = str(PROJECT_ROOT / "work" / "result_pretrained_backup")

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "work").mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Dataset:", DATASET_DIR)
print("Feature dictionary:", FEATURE_DICT)
print("Feature size (GB):", round(Path(FEATURE_DICT).stat().st_size / 1024**3, 2))
print("Windows safety: DataLoader workers are fixed at 0 to avoid duplicating the 11 GB feature dictionary.")

## 2. Repository and dataset setup

Use the local official EMS repository and prepare writable Windows paths.

In [ ]:
import os
import sys
from pathlib import Path

required_repo_files = [
    "MSNet.py",
    "val.py",
    "test.py",
    "data_process/split_fix.py",
    "data_process/metrics_val.py",
    "data_process/metrics_ostu.py",
]

print("Using the local official EMS repository:", REPO_DIR)
for relative in required_repo_files:
    path = Path(REPO_DIR) / relative
    print(relative, "->", path.exists())
    assert path.exists(), f"Missing EMS source file: {path}"

venv_python = PROJECT_ROOT / ".venv" / "Scripts" / "python.exe"
if venv_python.exists() and Path(sys.executable).resolve() != venv_python.resolve():
    print("WARNING: select the 'RVIF EMS (Windows GPU)' kernel before continuing.")

In [ ]:
import os
import shutil
from pathlib import Path

RESULT_DIR = str(PROJECT_ROOT / "work" / "msnet_results")

required_paths = {
    "RINet feature dictionary": Path(FEATURE_DICT),
    "pretrained model directory": Path(MODEL_SRC),
    "EMS dataset": Path(DATASET_DIR),
    "official EMS repository": Path(REPO_DIR),
    "fold workbook": Path(DATASET_DIR) / "Train_Valid.xlsx",
}
for name, path in required_paths.items():
    print(name, "->", path, "->", path.exists())
    assert path.exists(), f"Missing {name}: {path}"

# Copy only the small checkpoints to a writable result directory. The 11 GB
# feature dictionary and the dataset remain in place and are never duplicated.
for set_name in ["Set_0", "Set_1", "Set_2", "Set_3"]:
    source_weight = Path(MODEL_SRC) / set_name / "best_model.pt"
    target_dir = Path(RESULT_DIR) / set_name
    target_dir.mkdir(parents=True, exist_ok=True)
    target_weight = target_dir / "best_model.pt"
    assert source_weight.exists(), f"Missing pretrained checkpoint: {source_weight}"
    if not target_weight.exists() or target_weight.stat().st_size != source_weight.stat().st_size:
        shutil.copy2(source_weight, target_weight)

print("Local setup completed without copying the dataset or feature dictionary.")
print("Writable MSNet results:", RESULT_DIR)

## 3. MSNet baseline reproduction

Generate the official TXT inputs, run pretrained MSNet validation for four folds, and save baseline probabilities.

In [ ]:
import os
import sys
import glob
import subprocess
from pathlib import Path

TXT_DIR = str(Path(DATASET_DIR) / "TXT")
expected_txt_count = 208 * 100
existing_txt = list(Path(TXT_DIR).glob("*.txt")) if Path(TXT_DIR).exists() else []

if len(existing_txt) != expected_txt_count:
    print(f"Generating fixation TXT files ({len(existing_txt)}/{expected_txt_count} currently present)...")
    cmd = [
        sys.executable,
        str(Path(REPO_DIR) / "data_process" / "split_fix.py"),
        "--dataset_dir", DATASET_DIR,
    ]
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    print("TXT inputs already complete; skipping regeneration.")

txt_files = sorted(glob.glob(str(Path(TXT_DIR) / "*.txt")))
assert len(txt_files) == expected_txt_count, (
    f"Expected {expected_txt_count} TXT files, found {len(txt_files)}"
)
print("TXT directory:", TXT_DIR)
print("Number of TXT files:", len(txt_files))
print("First 10 TXT files:")
for path in txt_files[:10]:
    print(path)

In [ ]:
import os
import sys
import shutil
import glob
import subprocess
from pathlib import Path

RESULT_DIR = str(PROJECT_ROOT / "work" / "msnet_results")
BASELINE_BACKUP_DIR = str(PROJECT_ROOT / "work" / "result_pretrained_backup")

for set_name in ["Set_0", "Set_1", "Set_2", "Set_3"]:
    print(f"========== VALIDATE PRETRAINED {set_name} ==========")
    cmd = [
        sys.executable,
        str(Path(REPO_DIR) / "val.py"),
        "--dataset_dir", DATASET_DIR,
        "--feature_dict", FEATURE_DICT,
        "--result_root", RESULT_DIR,
        "--set_name", set_name,
        "--no_workers", "0",
    ]
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

subprocess.run(
    [sys.executable, str(Path(REPO_DIR) / "data_process" / "metrics_ostu.py"), "--result_root", RESULT_DIR],
    cwd=PROJECT_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, str(Path(REPO_DIR) / "data_process" / "metrics_val.py"), "--result_root", RESULT_DIR],
    cwd=PROJECT_ROOT,
    check=True,
)

summary_path = Path(RESULT_DIR) / "val.txt"
print("\nValidation summary:")
print(summary_path.read_text(encoding="utf-8"))

if Path(BASELINE_BACKUP_DIR).exists():
    shutil.rmtree(BASELINE_BACKUP_DIR)
shutil.copytree(RESULT_DIR, BASELINE_BACKUP_DIR)

print("Backed up baseline results to:", BASELINE_BACKUP_DIR)
for path in sorted(glob.glob(str(Path(BASELINE_BACKUP_DIR) / "Set_*" / "prob_val.txt"))):
    print(path)

## 4. Split metadata and raw fixation inspection

Create the subject-to-fold mapping and inspect the raw fixation files used for behavioral feature engineering.

In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
from pathlib import Path

REPO_DIR = str(PROJECT_ROOT / "EMS_repo")
DATASET_DIR = str(PROJECT_ROOT / "EMS")
TXT_DIR = f"{DATASET_DIR}/TXT"
TRAIN_VALID_XLSX = f"{DATASET_DIR}/Train_Valid.xlsx"
BASELINE_RESULT_DIR = str(PROJECT_ROOT / "work" / "result_pretrained_backup")
OUT_DIR = str(PROJECT_ROOT / "outputs" / "improve_behavioral")

os.makedirs(OUT_DIR, exist_ok=True)

print("Check paths:")
for p in [REPO_DIR, DATASET_DIR, TXT_DIR, TRAIN_VALID_XLSX, BASELINE_RESULT_DIR, OUT_DIR]:
    print(p, "->", os.path.exists(p))

assert os.path.exists(TRAIN_VALID_XLSX), "Missing Train_Valid.xlsx"
assert os.path.exists(TXT_DIR), "Missing TXT folder"
assert os.path.exists(BASELINE_RESULT_DIR), "Missing result_pretrained_backup"

def subject_to_int(x):
    """
    Convert subject IDs such as 59, '059', or 'Subject_059' to integer form.
    """
    nums = re.findall(r"\d+", str(x))
    if not nums:
        return None
    return int(nums[0])

def label_from_subject_int(subject_int):
    """
    Label subjects using the EMS repository convention: HC IDs are below 200 and SZ IDs are 200 or above.
    """
    return 1 if subject_int >= 200 else 0

df_split = pd.read_excel(TRAIN_VALID_XLSX)

print("\nTrain_Valid.xlsx shape:", df_split.shape)
print("Columns:")
print(df_split.columns.tolist())

print("\nPreview Train_Valid.xlsx:")
display(df_split.head())

set_cols = [c for c in df_split.columns if str(c).startswith("Set_")]
print("\nDetected set columns:", set_cols)

subject_to_set = {}
rows = []

for set_name in set_cols:
    for v in df_split[set_name].dropna().values:
        sid = subject_to_int(v)
        if sid is None:
            continue
        
        subject_to_set[sid] = set_name
        rows.append({
            "subject_int": sid,
            "subject_id": f"{sid:03d}",
            "set_name": set_name,
            "label": label_from_subject_int(sid)
        })

split_map = (
    pd.DataFrame(rows)
    .drop_duplicates("subject_int")
    .sort_values("subject_int")
    .reset_index(drop=True)
)

print("\nNumber of train/valid subjects:", len(split_map))

print("\nSubjects per fold:")
display(split_map["set_name"].value_counts().sort_index())

print("\nLabels overall:")
display(split_map["label"].value_counts().sort_index())

print("\nLabels per fold:")
display(pd.crosstab(split_map["set_name"], split_map["label"]))

print("\nPreview split_map:")
display(split_map.head(20))

split_map_path = f"{OUT_DIR}/split_map.csv"
split_map.to_csv(split_map_path, index=False)
print("\nSaved split_map to:", split_map_path)

In [ ]:
import os
import glob
import pandas as pd
from pathlib import Path

RAW_FIX_DIR = f"{DATASET_DIR}/Train_Valid/Fixations"

print("RAW_FIX_DIR:", RAW_FIX_DIR)
print("Exists:", os.path.exists(RAW_FIX_DIR))

raw_files = []
for ext in ["*.xlsx", "*.xls", "*.csv", "*.txt"]:
    raw_files.extend(glob.glob(f"{RAW_FIX_DIR}/**/{ext}", recursive=True))

raw_files = sorted(raw_files)

print("\nNumber of raw fixation files:", len(raw_files))
print("First 20 raw files:")
for p in raw_files[:20]:
    print(p)

def read_any_table(path, nrows=10):
    ext = Path(path).suffix.lower()
    
    try:
        if ext in [".xlsx", ".xls"]:
            return pd.read_excel(path, nrows=nrows)
        
        if ext == ".csv":
            return pd.read_csv(path, nrows=nrows)
        
        if ext == ".txt":
            try:
                return pd.read_csv(path, sep="\t", nrows=nrows)
            except Exception:
                try:
                    return pd.read_csv(path, sep=",", nrows=nrows)
                except Exception:
                    return pd.read_csv(path, header=None, nrows=nrows)
    except Exception as e:
        return None

print("\nInspect first readable files:")
shown = 0

for p in raw_files[:50]:
    df_sample = read_any_table(p, nrows=10)
    
    if df_sample is None or df_sample.empty:
        continue
    
    print("\n" + "=" * 80)
    print("FILE:", p)
    print("Shape sample:", df_sample.shape)
    print("Columns:")
    print(df_sample.columns.tolist())
    print("\nPreview:")
    display(df_sample.head(10))
    
    shown += 1
    if shown >= 3:
        break

if shown == 0:
    print("No readable raw fixation file was found. Check RAW_FIX_DIR.")

## 5. Behavioral feature engineering

Extract stimulus-level duration, pupil, and scanpath features, then aggregate them into subject-level features.

In [ ]:
import os
import re
import glob
import math
import numpy as np
import pandas as pd
from pathlib import Path

RAW_FIX_DIR = f"{DATASET_DIR}/Train_Valid/Fixations"
OUT_DIR = str(PROJECT_ROOT / "outputs" / "improve_behavioral")

SCREEN_W = 1024
SCREEN_H = 768
CENTER_X = SCREEN_W / 2
CENTER_Y = SCREEN_H / 2

def safe_std(x):
    x = np.asarray(x, dtype=float)
    if len(x) <= 1:
        return 0.0
    return float(np.std(x, ddof=0))

def safe_cv(mean, std):
    if mean is None or np.isnan(mean) or abs(mean) < 1e-8:
        return 0.0
    return float(std / mean)

def parse_subject_from_filename(path):
    """
    Raw fixation files are named by subject ID, for example 000.xlsx or 001.xlsx.
    """
    stem = Path(path).stem
    nums = re.findall(r"\d+", stem)
    if not nums:
        return None
    return int(nums[0])

def compute_one_stimulus_features(g):
    """
    Compute stimulus-level features from all fixations of one subject on one image.
    """
    out = {}
    
    g = g.sort_values("FIX_INDEX").reset_index(drop=True)
    
    fix_index = pd.to_numeric(g["FIX_INDEX"], errors="coerce")
    duration = pd.to_numeric(g["FIX_DURATION"], errors="coerce")
    x = pd.to_numeric(g["FIX_X"], errors="coerce")
    y = pd.to_numeric(g["FIX_Y"], errors="coerce")
    pupil = pd.to_numeric(g["FIX_PUPIL"], errors="coerce")
    
    df = pd.DataFrame({
        "fix_index": fix_index,
        "duration": duration,
        "x": x,
        "y": y,
        "pupil": pupil
    })
    
    df = df.dropna(subset=["fix_index", "x", "y"])
    df = df[
        (df["x"] >= 0) & (df["x"] < SCREEN_W) &
        (df["y"] >= 0) & (df["y"] < SCREEN_H)
    ]
    df = df.sort_values("fix_index").reset_index(drop=True)
    
    n = len(df)
    out["fixation_count"] = n
    
    if n == 0:
        return out
    
    d = df["duration"].dropna().values.astype(float)
    
    if len(d) > 0:
        d_mean = float(np.mean(d))
        d_std = safe_std(d)
        
        out["duration_mean"] = d_mean
        out["duration_std"] = d_std
        out["duration_cv"] = safe_cv(d_mean, d_std)
        out["duration_median"] = float(np.median(d))
        out["duration_min"] = float(np.min(d))
        out["duration_max"] = float(np.max(d))
        out["duration_total"] = float(np.sum(d))
        out["duration_q25"] = float(np.percentile(d, 25))
        out["duration_q75"] = float(np.percentile(d, 75))
        out["duration_iqr"] = out["duration_q75"] - out["duration_q25"]
        out["duration_first_fix"] = float(d[0])
        out["duration_first_5_mean"] = float(np.mean(d[:5]))
    else:
        for k in [
            "duration_mean", "duration_std", "duration_cv", "duration_median",
            "duration_min", "duration_max", "duration_total", "duration_q25",
            "duration_q75", "duration_iqr", "duration_first_fix", "duration_first_5_mean"
        ]:
            out[k] = np.nan
    
    p = df["pupil"].dropna().values.astype(float)
    
    if len(p) > 0:
        p_mean = float(np.mean(p))
        p_std = safe_std(p)
        
        out["pupil_mean"] = p_mean
        out["pupil_std"] = p_std
        out["pupil_cv"] = safe_cv(p_mean, p_std)
        out["pupil_median"] = float(np.median(p))
        out["pupil_min"] = float(np.min(p))
        out["pupil_max"] = float(np.max(p))
        out["pupil_q25"] = float(np.percentile(p, 25))
        out["pupil_q75"] = float(np.percentile(p, 75))
        out["pupil_iqr"] = out["pupil_q75"] - out["pupil_q25"]
        out["pupil_first_fix"] = float(p[0])
        out["pupil_first_5_mean"] = float(np.mean(p[:5]))
        
        delta_first = p - p[0]
        out["pupil_delta_first_mean"] = float(np.mean(delta_first))
        out["pupil_delta_first_std"] = safe_std(delta_first)
        out["pupil_delta_first_abs_mean"] = float(np.mean(np.abs(delta_first)))
        
        if len(p) >= 2:
            delta_step = np.diff(p)
            out["pupil_delta_step_mean"] = float(np.mean(delta_step))
            out["pupil_delta_step_std"] = safe_std(delta_step)
            out["pupil_delta_step_abs_mean"] = float(np.mean(np.abs(delta_step)))
        else:
            out["pupil_delta_step_mean"] = 0.0
            out["pupil_delta_step_std"] = 0.0
            out["pupil_delta_step_abs_mean"] = 0.0
    else:
        for k in [
            "pupil_mean", "pupil_std", "pupil_cv", "pupil_median", "pupil_min",
            "pupil_max", "pupil_q25", "pupil_q75", "pupil_iqr", "pupil_first_fix",
            "pupil_first_5_mean", "pupil_delta_first_mean", "pupil_delta_first_std",
            "pupil_delta_first_abs_mean", "pupil_delta_step_mean", "pupil_delta_step_std",
            "pupil_delta_step_abs_mean"
        ]:
            out[k] = np.nan
    
    out["pupil_missing_ratio"] = float(df["pupil"].isna().mean())
    
    xs = df["x"].values.astype(float)
    ys = df["y"].values.astype(float)
    
    out["x_mean"] = float(np.mean(xs))
    out["y_mean"] = float(np.mean(ys))
    out["x_std"] = safe_std(xs)
    out["y_std"] = safe_std(ys)
    
    out["x_dispersion"] = float(np.max(xs) - np.min(xs))
    out["y_dispersion"] = float(np.max(ys) - np.min(ys))
    out["xy_dispersion_area"] = out["x_dispersion"] * out["y_dispersion"]
    
    dist_center = np.sqrt((xs - CENTER_X) ** 2 + (ys - CENTER_Y) ** 2)
    out["distance_to_center_mean"] = float(np.mean(dist_center))
    out["distance_to_center_std"] = safe_std(dist_center)
    out["distance_to_center_max"] = float(np.max(dist_center))
    
    if n >= 2:
        dx = np.diff(xs)
        dy = np.diff(ys)
        sacc_amp = np.sqrt(dx ** 2 + dy ** 2)
        
        out["scanpath_length"] = float(np.sum(sacc_amp))
        out["saccade_amplitude_mean"] = float(np.mean(sacc_amp))
        out["saccade_amplitude_std"] = safe_std(sacc_amp)
        out["saccade_amplitude_median"] = float(np.median(sacc_amp))
        out["saccade_amplitude_max"] = float(np.max(sacc_amp))
    else:
        out["scanpath_length"] = 0.0
        out["saccade_amplitude_mean"] = 0.0
        out["saccade_amplitude_std"] = 0.0
        out["saccade_amplitude_median"] = 0.0
        out["saccade_amplitude_max"] = 0.0
    
    return out

raw_files = sorted(glob.glob(f"{RAW_FIX_DIR}/*.xlsx"))

stim_rows = []

for path in raw_files:
    sid = parse_subject_from_filename(path)
    
    if sid is None or sid not in subject_to_set:
        continue
    
    df_raw = pd.read_excel(path)
    
    required_cols = ["IMAGE", "FIX_INDEX", "FIX_DURATION", "FIX_X", "FIX_Y", "FIX_PUPIL"]
    missing = [c for c in required_cols if c not in df_raw.columns]
    if missing:
        print("Missing columns in", path, ":", missing)
        continue
    
    for image_name, g in df_raw.groupby("IMAGE"):
        feat = compute_one_stimulus_features(g)
        
        feat["subject_int"] = sid
        feat["subject_id"] = f"{sid:03d}"
        feat["set_name"] = subject_to_set[sid]
        feat["label"] = label_from_subject_int(sid)
        feat["image"] = image_name
        
        stim_rows.append(feat)

stim_behavior_features = pd.DataFrame(stim_rows)

meta_cols = ["subject_int", "subject_id", "set_name", "label", "image"]
other_cols = [c for c in stim_behavior_features.columns if c not in meta_cols]
stim_behavior_features = stim_behavior_features[meta_cols + other_cols]

stim_path = f"{OUT_DIR}/stim_behavior_features.csv"
stim_behavior_features.to_csv(stim_path, index=False)

print("Stimulus-level behavior features shape:", stim_behavior_features.shape)
print("Saved to:", stim_path)

print("\nNumber of subjects:", stim_behavior_features["subject_int"].nunique())
print("Number of unique images:", stim_behavior_features["image"].nunique())
print("Rows per subject summary:")
display(stim_behavior_features.groupby("subject_id")["image"].nunique().describe())

print("\nColumns:")
print(stim_behavior_features.columns.tolist())

print("\nPreview:")
display(stim_behavior_features.head())

In [ ]:
import os
import numpy as np
import pandas as pd

OUT_DIR = str(PROJECT_ROOT / "outputs" / "improve_behavioral")

stim_path = f"{OUT_DIR}/stim_behavior_features.csv"
if "stim_behavior_features" not in globals():
    stim_behavior_features = pd.read_csv(stim_path)

print("Loaded stim_behavior_features:", stim_behavior_features.shape)

meta_cols = ["subject_int", "subject_id", "set_name", "label", "image"]
numeric_cols = [c for c in stim_behavior_features.columns if c not in meta_cols]

print("Number of numeric stimulus features:", len(numeric_cols))

agg_funcs = ["mean", "std", "median", "min", "max"]

subject_agg = stim_behavior_features.groupby("subject_int")[numeric_cols].agg(agg_funcs)

subject_agg.columns = [f"{col}_{func}" for col, func in subject_agg.columns]
subject_agg = subject_agg.reset_index()

subject_meta = (
    stim_behavior_features[["subject_int", "subject_id", "set_name", "label"]]
    .drop_duplicates("subject_int")
    .copy()
)

num_valid_images = (
    stim_behavior_features.groupby("subject_int")["image"]
    .nunique()
    .reset_index()
    .rename(columns={"image": "num_valid_images"})
)

num_total_fixations = (
    stim_behavior_features.groupby("subject_int")["fixation_count"]
    .sum()
    .reset_index()
    .rename(columns={"fixation_count": "num_total_fixations"})
)

mean_fix_per_image = (
    stim_behavior_features.groupby("subject_int")["fixation_count"]
    .mean()
    .reset_index()
    .rename(columns={"fixation_count": "mean_fixations_per_image"})
)

subject_behavior_features = subject_meta.merge(subject_agg, on="subject_int", how="left")
subject_behavior_features = subject_behavior_features.merge(num_valid_images, on="subject_int", how="left")
subject_behavior_features = subject_behavior_features.merge(num_total_fixations, on="subject_int", how="left")
subject_behavior_features = subject_behavior_features.merge(mean_fix_per_image, on="subject_int", how="left")

subject_behavior_features = subject_behavior_features.sort_values("subject_int").reset_index(drop=True)

subject_path = f"{OUT_DIR}/subject_behavior_features.csv"
subject_behavior_features.to_csv(subject_path, index=False)

print("Subject-level behavior features shape:", subject_behavior_features.shape)
print("Saved to:", subject_path)

print("\nNumber of subjects:", subject_behavior_features["subject_int"].nunique())

print("\nLabels:")
display(subject_behavior_features["label"].value_counts().sort_index())

print("\nSets:")
display(subject_behavior_features["set_name"].value_counts().sort_index())

print("\nValid images summary:")
display(subject_behavior_features["num_valid_images"].describe())

print("\nMissing value ratio top 20:")
feature_cols_tmp = [c for c in subject_behavior_features.columns if c not in ["subject_int", "subject_id", "set_name", "label"]]
missing_ratio = subject_behavior_features[feature_cols_tmp].isna().mean().sort_values(ascending=False)
display(missing_ratio.head(20))

print("\nPreview subject-level features:")
display(subject_behavior_features.head())

## 6. Feature groups and exploratory statistics

Define the candidate behavioral feature groups used in the controlled experiments.

In [ ]:
import os
import numpy as np
import pandas as pd

OUT_DIR = str(PROJECT_ROOT / "outputs" / "improve_behavioral")
subject_path = f"{OUT_DIR}/subject_behavior_features.csv"

if "subject_behavior_features" not in globals():
    subject_behavior_features = pd.read_csv(subject_path)

print("Subject-level feature table:", subject_behavior_features.shape)

meta_cols = ["subject_int", "subject_id", "set_name", "label"]
all_feature_cols = [c for c in subject_behavior_features.columns if c not in meta_cols]

duration_cols = [
    c for c in all_feature_cols
    if c.startswith("duration_")
]

pupil_cols = [
    c for c in all_feature_cols
    if c.startswith("pupil_")
]

scanpath_keywords = [
    "fixation_count",
    "scanpath",
    "saccade",
    "x_mean", "y_mean",
    "x_std", "y_std",
    "x_dispersion", "y_dispersion",
    "xy_dispersion_area",
    "distance_to_center",
    "num_valid_images",
    "num_total_fixations",
    "mean_fixations_per_image"
]

scanpath_cols = [
    c for c in all_feature_cols
    if any(k in c for k in scanpath_keywords)
]

duration_cols = sorted(list(set(duration_cols)))
pupil_cols = sorted(list(set(pupil_cols)))
scanpath_cols = sorted(list(set(scanpath_cols)))

feature_sets = {
    "duration_only": duration_cols,
    "pupil_only": pupil_cols,
    "scanpath_only": scanpath_cols,
    "duration_pupil": sorted(list(set(duration_cols + pupil_cols))),
    "duration_scanpath": sorted(list(set(duration_cols + scanpath_cols))),
    "pupil_scanpath": sorted(list(set(pupil_cols + scanpath_cols))),
    "all_behavior": sorted(list(set(duration_cols + pupil_cols + scanpath_cols))),
}

print("\nFeature set sizes:")
for name, cols in feature_sets.items():
    print(f"{name}: {len(cols)}")

print("\nSample duration cols:")
print(duration_cols[:20])

print("\nSample pupil cols:")
print(pupil_cols[:20])

print("\nSample scanpath cols:")
print(scanpath_cols[:20])

empty_sets = [k for k, v in feature_sets.items() if len(v) == 0]
print("\nEmpty feature sets:", empty_sets)

assert len(duration_cols) > 0, "duration_cols is empty"
assert len(pupil_cols) > 0, "pupil_cols is empty"
assert len(scanpath_cols) > 0, "scanpath_cols is empty"

eda_rows = []

for col in all_feature_cols:
    hc_vals = subject_behavior_features.loc[subject_behavior_features["label"] == 0, col]
    sz_vals = subject_behavior_features.loc[subject_behavior_features["label"] == 1, col]
    
    hc_mean = hc_vals.mean()
    sz_mean = sz_vals.mean()
    hc_std = hc_vals.std()
    sz_std = sz_vals.std()
    
    pooled_std = np.sqrt(((hc_std ** 2) + (sz_std ** 2)) / 2)
    if pooled_std is None or np.isnan(pooled_std) or pooled_std == 0:
        cohen_d = 0.0
    else:
        cohen_d = (sz_mean - hc_mean) / pooled_std
    
    eda_rows.append({
        "feature": col,
        "hc_mean": hc_mean,
        "sz_mean": sz_mean,
        "diff_sz_minus_hc": sz_mean - hc_mean,
        "abs_diff": abs(sz_mean - hc_mean),
        "cohen_d": cohen_d,
        "abs_cohen_d": abs(cohen_d),
        "missing_ratio": subject_behavior_features[col].isna().mean()
    })

eda_df = pd.DataFrame(eda_rows).sort_values("abs_cohen_d", ascending=False).reset_index(drop=True)

eda_path = f"{OUT_DIR}/eda_group_mean_diff.csv"
eda_df.to_csv(eda_path, index=False)

print("\nSaved EDA to:", eda_path)

print("\nTop 30 features by abs(Cohen's d):")
display(eda_df.head(30))

import json

feature_sets_path = f"{OUT_DIR}/feature_sets.json"
with open(feature_sets_path, "w") as f:
    json.dump(feature_sets, f, indent=2)

print("\nSaved feature sets to:", feature_sets_path)

In [ ]:
import os
import json
import pandas as pd
import numpy as np

OUT_DIR = str(PROJECT_ROOT / "outputs" / "improve_behavioral")
subject_path = f"{OUT_DIR}/subject_behavior_features.csv"

if "subject_behavior_features" not in globals():
    subject_behavior_features = pd.read_csv(subject_path)

meta_cols = ["subject_int", "subject_id", "set_name", "label"]
all_feature_cols = [c for c in subject_behavior_features.columns if c not in meta_cols]

duration_cols = sorted([
    c for c in all_feature_cols
    if c.startswith("duration_")
])

pupil_cols = sorted([
    c for c in all_feature_cols
    if c.startswith("pupil_")
])

scanpath_base_names = [
    "fixation_count",
    "x_mean",
    "y_mean",
    "x_std",
    "y_std",
    "x_dispersion",
    "y_dispersion",
    "xy_dispersion_area",
    "distance_to_center_mean",
    "distance_to_center_std",
    "distance_to_center_max",
    "scanpath_length",
    "saccade_amplitude_mean",
    "saccade_amplitude_std",
    "saccade_amplitude_median",
    "saccade_amplitude_max",
    "num_valid_images",
    "num_total_fixations",
    "mean_fixations_per_image",
]

def belongs_to_base_feature(col, base_names):
    for base in base_names:
        if col == base:
            return True
        if col.startswith(base + "_"):
            return True
    return False

scanpath_cols = sorted([
    c for c in all_feature_cols
    if belongs_to_base_feature(c, scanpath_base_names)
])

duration_in_scanpath = [c for c in scanpath_cols if c.startswith("duration_")]
pupil_in_scanpath = [c for c in scanpath_cols if c.startswith("pupil_")]

print("duration_cols:", len(duration_cols))
print("pupil_cols:", len(pupil_cols))
print("scanpath_cols:", len(scanpath_cols))

print("\nUnexpected duration features inside scanpath:", duration_in_scanpath[:20])
print("Unexpected pupil features inside scanpath:", pupil_in_scanpath[:20])

assert len(duration_in_scanpath) == 0, "scanpath_cols still contains duration features"
assert len(pupil_in_scanpath) == 0, "scanpath_cols still contains pupil features"

feature_sets = {
    "duration_only": duration_cols,
    "pupil_only": pupil_cols,
    "scanpath_only": scanpath_cols,
    "duration_pupil": sorted(list(set(duration_cols + pupil_cols))),
    "duration_scanpath": sorted(list(set(duration_cols + scanpath_cols))),
    "pupil_scanpath": sorted(list(set(pupil_cols + scanpath_cols))),
    "all_behavior": sorted(list(set(duration_cols + pupil_cols + scanpath_cols))),
}

print("\nClean feature set sizes:")
for name, cols in feature_sets.items():
    print(f"{name}: {len(cols)}")

print("\nSample clean scanpath cols:")
print(scanpath_cols[:40])

feature_sets_path = f"{OUT_DIR}/feature_sets_clean.json"
with open(feature_sets_path, "w") as f:
    json.dump(feature_sets, f, indent=2)

print("\nSaved clean feature sets to:", feature_sets_path)

## 7. Baseline metrics and behavioral models

Load MSNet probabilities, compute benchmark metrics, and train behavioral-only models using the four-fold protocol.

In [ ]:
import os
import re
import glob
import cv2
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

REPO_DIR = str(PROJECT_ROOT / "EMS_repo")
BASELINE_RESULT_DIR = str(PROJECT_ROOT / "work" / "result_pretrained_backup")
OUT_DIR = str(PROJECT_ROOT / "outputs" / "improve_behavioral")

print("BASELINE_RESULT_DIR:", BASELINE_RESULT_DIR)
print("Exists:", os.path.exists(BASELINE_RESULT_DIR))

def subject_to_int(x):
    nums = re.findall(r"\d+", str(x))
    if not nums:
        return None
    return int(nums[0])

def label_from_subject_int(subject_int):
    return 1 if subject_int >= 200 else 0

def read_msnet_prob_val(result_dir):
    rows = []
    
    for set_name in ["Set_0", "Set_1", "Set_2", "Set_3"]:
        path = f"{result_dir}/{set_name}/prob_val.txt"
        assert os.path.exists(path), f"Missing {path}"
        
        with open(path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                
                parts = line.split(",")
                
                subject_raw = parts[0]
                prob = float(parts[1])
                sid = subject_to_int(subject_raw)
                
                rows.append({
                    "subject_int": sid,
                    "subject_id": f"{sid:03d}",
                    "set_name": set_name,
                    "label": label_from_subject_int(sid),
                    "msnet_prob": prob
                })
    
    df = pd.DataFrame(rows)
    df = df.sort_values(["set_name", "subject_int"]).reset_index(drop=True)
    return df

def otsu_threshold_probs(probs):
    """
    Compute the Otsu-style threshold used by the EMS repository.
    """
    probs = np.asarray(probs, dtype=float)
    vals = np.clip(probs * 255.0, 0, 255).astype(np.uint8)
    thr, _ = cv2.threshold(vals, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return float(thr / 255.0)

def compute_metrics(y_true, y_prob, threshold=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    
    if threshold is None:
        threshold = otsu_threshold_probs(y_prob)
    
    y_pred = (y_prob >= threshold).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    
    acc = accuracy_score(y_true, y_pred)
    sensitivity = recall_score(y_true, y_pred, zero_division=0)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    precision = precision_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob)
    
    return {
        "threshold": threshold,
        "acc": acc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "auc": auc,
        "precision": precision,
        "f1_score": f1,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }

msnet_probs = read_msnet_prob_val(BASELINE_RESULT_DIR)

print("\nMSNet probability table shape:", msnet_probs.shape)

print("\nPreview:")
display(msnet_probs.head())

print("\nSubjects per fold:")
display(msnet_probs["set_name"].value_counts().sort_index())

print("\nLabels:")
display(msnet_probs["label"].value_counts().sort_index())

print("\nLabels per fold:")
display(pd.crosstab(msnet_probs["set_name"], msnet_probs["label"]))

baseline_metrics = compute_metrics(msnet_probs["label"], msnet_probs["msnet_prob"])

print("\nRecomputed MSNet baseline metrics from prob_val:")
for k, v in baseline_metrics.items():
    print(f"{k}: {v}")

msnet_prob_path = f"{OUT_DIR}/msnet_prob_val.csv"
msnet_probs.to_csv(msnet_prob_path, index=False)

print("\nSaved MSNet prob to:", msnet_prob_path)

In [ ]:
import os
import numpy as np
import pandas as pd

OUT_DIR = str(PROJECT_ROOT / "outputs" / "improve_behavioral")

msnet_prob_path = f"{OUT_DIR}/msnet_prob_val.csv"
if "msnet_probs" not in globals():
    msnet_probs = pd.read_csv(msnet_prob_path)

def compute_foldwise_metrics(df, prob_col, label_col="label", set_col="set_name"):
    """
    Compute fold-wise metrics for Set_0..Set_3 and average them.
    """
    rows = []
    
    for set_name in ["Set_0", "Set_1", "Set_2", "Set_3"]:
        fold_df = df[df[set_col] == set_name].copy()
        
        assert len(fold_df) > 0, f"No data found for {set_name}"
        
        m = compute_metrics(
            y_true=fold_df[label_col].values,
            y_prob=fold_df[prob_col].values,
            threshold=None
        )
        
        row = {
            "set_name": set_name,
            **m
        }
        rows.append(row)
    
    fold_metrics = pd.DataFrame(rows)
    
    metric_cols = [
        "threshold",
        "acc",
        "sensitivity",
        "specificity",
        "auc",
        "precision",
        "f1_score"
    ]
    
    avg_row = {"set_name": "average"}
    for c in metric_cols:
        avg_row[c] = fold_metrics[c].mean()
    
    for c in ["tn", "fp", "fn", "tp"]:
        avg_row[c] = np.nan
    
    fold_metrics = pd.concat(
        [fold_metrics, pd.DataFrame([avg_row])],
        ignore_index=True
    )
    
    return fold_metrics

baseline_foldwise_metrics = compute_foldwise_metrics(msnet_probs, prob_col="msnet_prob")

baseline_foldwise_path = f"{OUT_DIR}/baseline_msnet_foldwise_metrics.csv"
baseline_foldwise_metrics.to_csv(baseline_foldwise_path, index=False)

print("Baseline MSNet fold-wise metrics:")
display(baseline_foldwise_metrics)

print("\nSaved to:", baseline_foldwise_path)

print("\nAverage row:")
display(baseline_foldwise_metrics[baseline_foldwise_metrics["set_name"] == "average"])

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

OUT_DIR = str(PROJECT_ROOT / "outputs" / "improve_behavioral")

subject_path = f"{OUT_DIR}/subject_behavior_features.csv"
feature_sets_path = f"{OUT_DIR}/feature_sets_clean.json"

if "subject_behavior_features" not in globals():
    subject_behavior_features = pd.read_csv(subject_path)

if "feature_sets" not in globals():
    with open(feature_sets_path, "r") as f:
        feature_sets = json.load(f)

print("Subject behavior features:", subject_behavior_features.shape)

print("\nFeature sets:")
for k, v in feature_sets.items():
    print(k, len(v))

def make_behavior_model(model_name):
    """
    Create a preprocessing and classifier pipeline. The imputer and scaler are fit only on the training folds.
    """
    if model_name == "logreg":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                solver="liblinear",
                random_state=42
            ))
        ])
    
    if model_name == "rf":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=500,
                max_depth=None,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            ))
        ])
    
    if model_name == "hgb":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", HistGradientBoostingClassifier(
                max_iter=300,
                learning_rate=0.03,
                max_leaf_nodes=15,
                l2_regularization=0.01,
                random_state=42
            ))
        ])
    
    raise ValueError(f"Unknown model_name: {model_name}")

def run_behavior_cv(feature_cols, model_name):
    """
    Train and validate using the four-fold subject split.
    """
    pred_rows = []
    
    for val_set in ["Set_0", "Set_1", "Set_2", "Set_3"]:
        train_df = subject_behavior_features[subject_behavior_features["set_name"] != val_set].copy()
        val_df = subject_behavior_features[subject_behavior_features["set_name"] == val_set].copy()
        
        X_train = train_df[feature_cols]
        y_train = train_df["label"].astype(int)
        
        X_val = val_df[feature_cols]
        y_val = val_df["label"].astype(int)
        
        model = make_behavior_model(model_name)
        model.fit(X_train, y_train)
        
        behavior_prob = model.predict_proba(X_val)[:, 1]
        
        for sid, sid_str, set_name, label, prob in zip(
            val_df["subject_int"],
            val_df["subject_id"],
            val_df["set_name"],
            y_val,
            behavior_prob
        ):
            pred_rows.append({
                "subject_int": int(sid),
                "subject_id": sid_str,
                "set_name": set_name,
                "label": int(label),
                "behavior_prob": float(prob)
            })
    
    pred_df = (
        pd.DataFrame(pred_rows)
        .sort_values(["set_name", "subject_int"])
        .reset_index(drop=True)
    )
    
    foldwise_metrics = compute_foldwise_metrics(pred_df, prob_col="behavior_prob")
    
    return pred_df, foldwise_metrics

model_names = ["logreg", "rf", "hgb"]

behavior_summary_rows = []
behavior_pred_cache = {}
behavior_metrics_cache = {}

for feature_set_name, feature_cols in feature_sets.items():
    for model_name in model_names:
        print(f"\nRunning feature_set={feature_set_name}, model={model_name}, n_features={len(feature_cols)}")
        
        pred_df, foldwise_metrics = run_behavior_cv(feature_cols, model_name)
        
        key = f"{feature_set_name}__{model_name}"
        behavior_pred_cache[key] = pred_df
        behavior_metrics_cache[key] = foldwise_metrics
        
        avg = foldwise_metrics[foldwise_metrics["set_name"] == "average"].iloc[0].to_dict()
        
        behavior_summary_rows.append({
            "key": key,
            "feature_set": feature_set_name,
            "model": model_name,
            "n_features": len(feature_cols),
            "acc": avg["acc"],
            "sensitivity": avg["sensitivity"],
            "specificity": avg["specificity"],
            "auc": avg["auc"],
            "precision": avg["precision"],
            "f1_score": avg["f1_score"],
            "threshold_mean": avg["threshold"]
        })

behavior_only_metrics = (
    pd.DataFrame(behavior_summary_rows)
    .sort_values(["auc", "f1_score", "acc"], ascending=False)
    .reset_index(drop=True)
)

behavior_metrics_path = f"{OUT_DIR}/behavior_only_metrics.csv"
behavior_only_metrics.to_csv(behavior_metrics_path, index=False)

print("\nSaved behavior-only summary to:", behavior_metrics_path)

print("\nBehavior-only results sorted by AUC:")
display(behavior_only_metrics)

best_key = behavior_only_metrics.iloc[0]["key"]
best_pred = behavior_pred_cache[best_key]
best_foldwise = behavior_metrics_cache[best_key]

best_pred_path = f"{OUT_DIR}/best_behavior_prob_val.csv"
best_foldwise_path = f"{OUT_DIR}/best_behavior_foldwise_metrics.csv"

best_pred.to_csv(best_pred_path, index=False)
best_foldwise.to_csv(best_foldwise_path, index=False)

print("\nBest behavior key:", best_key)
print("Saved best behavior predictions to:", best_pred_path)
print("Saved best behavior foldwise metrics to:", best_foldwise_path)

print("\nBest behavior foldwise metrics:")
display(best_foldwise)

## 8. Probability-level fusion

Fuse MSNet and behavioral probabilities with nested alpha selection and choose the final model by average AUC.

In [ ]:
import os
import numpy as np
import pandas as pd

OUT_DIR = str(PROJECT_ROOT / "outputs" / "improve_behavioral")

assert "msnet_probs" in globals(), "Missing msnet_probs. Run the baseline probability cell first."
assert "behavior_pred_cache" in globals(), "Missing behavior_pred_cache. Run the behavioral model cell first."
assert "behavior_only_metrics" in globals(), "Missing behavior_only_metrics. Run the behavioral model cell first."

alpha_grid = np.round(np.arange(0.0, 1.0001, 0.05), 2)

def compute_foldwise_metrics_available(df, prob_col, label_col="label", set_col="set_name"):
    """
    Compute metrics on the folds present in the dataframe.
    """
    rows = []
    available_sets = sorted(df[set_col].unique().tolist())
    
    for set_name in available_sets:
        fold_df = df[df[set_col] == set_name].copy()
        
        if len(fold_df) == 0:
            continue
        
        m = compute_metrics(
            y_true=fold_df[label_col].values,
            y_prob=fold_df[prob_col].values,
            threshold=None
        )
        
        rows.append({
            "set_name": set_name,
            **m
        })
    
    fold_metrics = pd.DataFrame(rows)
    
    metric_cols = [
        "threshold",
        "acc",
        "sensitivity",
        "specificity",
        "auc",
        "precision",
        "f1_score"
    ]
    
    avg_row = {"set_name": "average"}
    for c in metric_cols:
        avg_row[c] = fold_metrics[c].mean()
    
    for c in ["tn", "fp", "fn", "tp"]:
        avg_row[c] = np.nan
    
    fold_metrics = pd.concat(
        [fold_metrics, pd.DataFrame([avg_row])],
        ignore_index=True
    )
    
    return fold_metrics

def evaluate_alpha_on_available_folds(df, alpha):
    """
    Evaluate one alpha on the provided folds.
    """
    tmp = df.copy()
    tmp["fused_prob"] = (
        alpha * tmp["msnet_prob"].values +
        (1 - alpha) * tmp["behavior_prob"].values
    )
    
    m = compute_foldwise_metrics_available(tmp, prob_col="fused_prob")
    avg = m[m["set_name"] == "average"].iloc[0].to_dict()
    return avg

def choose_alpha_on_training_folds(merged_df, val_set, alpha_values):
    """
    Select alpha on the other three folds using AUC, then F1, then accuracy as tie-breakers.
    """
    train_like_df = merged_df[merged_df["set_name"] != val_set].copy()
    
    assert len(train_like_df) > 0, "train_like_df is empty"
    assert val_set not in train_like_df["set_name"].unique(), "val_set leaked into alpha selection"
    
    rows = []
    
    for alpha in alpha_values:
        avg = evaluate_alpha_on_available_folds(train_like_df, alpha)
        
        rows.append({
            "alpha_msnet": alpha,
            "auc": avg["auc"],
            "f1_score": avg["f1_score"],
            "acc": avg["acc"],
            "sensitivity": avg["sensitivity"],
            "specificity": avg["specificity"],
            "precision": avg["precision"]
        })
    
    alpha_df = pd.DataFrame(rows).sort_values(
        ["auc", "f1_score", "acc"],
        ascending=False
    ).reset_index(drop=True)
    
    best_alpha = float(alpha_df.iloc[0]["alpha_msnet"])
    
    return best_alpha, alpha_df

def run_nested_alpha_fusion(behavior_pred_df, msnet_probs, alpha_values):
    """
    For each validation fold, select alpha on the remaining folds and evaluate on the held-out fold.
    """
    merged = msnet_probs.merge(
        behavior_pred_df[["subject_int", "set_name", "behavior_prob"]],
        on=["subject_int", "set_name"],
        how="inner"
    )
    
    assert len(merged) == len(msnet_probs), f"Merge missed subjects: {len(merged)} vs {len(msnet_probs)}"
    
    fused_rows = []
    alpha_rows = []
    
    for val_set in ["Set_0", "Set_1", "Set_2", "Set_3"]:
        best_alpha, alpha_search_df = choose_alpha_on_training_folds(
            merged_df=merged,
            val_set=val_set,
            alpha_values=alpha_values
        )
        
        val_df = merged[merged["set_name"] == val_set].copy()
        
        assert len(val_df) > 0, f"No validation data for {val_set}"
        
        val_df["alpha_msnet"] = best_alpha
        val_df["behavior_weight"] = 1 - best_alpha
        val_df["fused_prob"] = (
            best_alpha * val_df["msnet_prob"] +
            (1 - best_alpha) * val_df["behavior_prob"]
        )
        
        fused_rows.append(val_df)
        
        alpha_rows.append({
            "val_set": val_set,
            "chosen_alpha_msnet": best_alpha,
            "chosen_behavior_weight": 1 - best_alpha,
            "alpha_search_best_auc": float(alpha_search_df.iloc[0]["auc"]),
            "alpha_search_best_f1": float(alpha_search_df.iloc[0]["f1_score"]),
            "alpha_search_best_acc": float(alpha_search_df.iloc[0]["acc"])
        })
    
    fused_df = pd.concat(fused_rows, ignore_index=True)
    alpha_df = pd.DataFrame(alpha_rows)
    
    foldwise_metrics = compute_foldwise_metrics(fused_df, prob_col="fused_prob")
    
    return fused_df, foldwise_metrics, alpha_df

fusion_summary_rows = []
fusion_pred_cache = {}
fusion_metrics_cache = {}
fusion_alpha_cache = {}

for key, behavior_pred_df in behavior_pred_cache.items():
    print("Running fusion for:", key)
    
    fused_df, foldwise_metrics, alpha_df = run_nested_alpha_fusion(
        behavior_pred_df=behavior_pred_df,
        msnet_probs=msnet_probs,
        alpha_values=alpha_grid
    )
    
    fusion_pred_cache[key] = fused_df
    fusion_metrics_cache[key] = foldwise_metrics
    fusion_alpha_cache[key] = alpha_df
    
    avg = foldwise_metrics[foldwise_metrics["set_name"] == "average"].iloc[0].to_dict()
    
    feature_set, model_name = key.split("__")
    
    fusion_summary_rows.append({
        "key": key,
        "feature_set": feature_set,
        "model": model_name,
        "acc": avg["acc"],
        "sensitivity": avg["sensitivity"],
        "specificity": avg["specificity"],
        "auc": avg["auc"],
        "precision": avg["precision"],
        "f1_score": avg["f1_score"],
        "threshold_mean": avg["threshold"],
        "alpha_mean": alpha_df["chosen_alpha_msnet"].mean(),
        "behavior_weight_mean": alpha_df["chosen_behavior_weight"].mean()
    })

fusion_metrics_df = (
    pd.DataFrame(fusion_summary_rows)
    .sort_values(["auc", "f1_score", "acc"], ascending=False)
    .reset_index(drop=True)
)

fusion_metrics_path = f"{OUT_DIR}/fusion_metrics_nested_alpha.csv"
fusion_metrics_df.to_csv(fusion_metrics_path, index=False)

print("\nSaved fusion metrics to:", fusion_metrics_path)

print("\nFusion results sorted by AUC:")
display(fusion_metrics_df)

best_fusion_key = fusion_metrics_df.iloc[0]["key"]
best_fusion_pred = fusion_pred_cache[best_fusion_key]
best_fusion_foldwise = fusion_metrics_cache[best_fusion_key]
best_fusion_alpha = fusion_alpha_cache[best_fusion_key]

best_fusion_pred_path = f"{OUT_DIR}/best_fusion_prob_val.csv"
best_fusion_foldwise_path = f"{OUT_DIR}/best_fusion_foldwise_metrics.csv"
best_fusion_alpha_path = f"{OUT_DIR}/best_fusion_alpha_by_fold.csv"

best_fusion_pred.to_csv(best_fusion_pred_path, index=False)
best_fusion_foldwise.to_csv(best_fusion_foldwise_path, index=False)
best_fusion_alpha.to_csv(best_fusion_alpha_path, index=False)

print("\nBest fusion key:", best_fusion_key)
print("Saved best fusion predictions to:", best_fusion_pred_path)
print("Saved best fusion foldwise metrics to:", best_fusion_foldwise_path)
print("Saved best fusion alpha by fold to:", best_fusion_alpha_path)

print("\nBest fusion alpha by fold:")
display(best_fusion_alpha)

print("\nBest fusion foldwise metrics:")
display(best_fusion_foldwise)

print("\nBaseline average for comparison:")
display(baseline_foldwise_metrics[baseline_foldwise_metrics["set_name"] == "average"])

print("\nBest behavior-only average for comparison:")
display(best_foldwise[best_foldwise["set_name"] == "average"])

In [ ]:
import os
import pandas as pd
import numpy as np

OUT_DIR = str(PROJECT_ROOT / "outputs" / "improve_behavioral")

if "baseline_foldwise_metrics" not in globals():
    baseline_foldwise_metrics = pd.read_csv(f"{OUT_DIR}/baseline_msnet_foldwise_metrics.csv")

if "behavior_only_metrics" not in globals():
    behavior_only_metrics = pd.read_csv(f"{OUT_DIR}/behavior_only_metrics.csv")

if "fusion_metrics_df" not in globals():
    fusion_metrics_df = pd.read_csv(f"{OUT_DIR}/fusion_metrics_nested_alpha.csv")

if "best_foldwise" not in globals():
    best_foldwise = pd.read_csv(f"{OUT_DIR}/best_behavior_foldwise_metrics.csv")

if "best_fusion_foldwise" not in globals():
    best_fusion_foldwise = pd.read_csv(f"{OUT_DIR}/best_fusion_foldwise_metrics.csv")

baseline_avg = baseline_foldwise_metrics[baseline_foldwise_metrics["set_name"] == "average"].iloc[0]
best_behavior_avg = best_foldwise[best_foldwise["set_name"] == "average"].iloc[0]
best_fusion_avg = best_fusion_foldwise[best_fusion_foldwise["set_name"] == "average"].iloc[0]

best_behavior_row = behavior_only_metrics.iloc[0]
best_fusion_row = fusion_metrics_df.iloc[0]

comparison_rows = [
    {
        "method": "MSNet baseline",
        "feature_set": "RINet fixation features",
        "model": "official pretrained MSNet",
        "alpha_msnet": 1.0,
        "behavior_weight": 0.0,
        "acc": baseline_avg["acc"],
        "sensitivity": baseline_avg["sensitivity"],
        "specificity": baseline_avg["specificity"],
        "auc": baseline_avg["auc"],
        "precision": baseline_avg["precision"],
        "f1_score": baseline_avg["f1_score"],
    },
    {
        "method": "Behavior-only",
        "feature_set": best_behavior_row["feature_set"],
        "model": best_behavior_row["model"],
        "alpha_msnet": 0.0,
        "behavior_weight": 1.0,
        "acc": best_behavior_avg["acc"],
        "sensitivity": best_behavior_avg["sensitivity"],
        "specificity": best_behavior_avg["specificity"],
        "auc": best_behavior_avg["auc"],
        "precision": best_behavior_avg["precision"],
        "f1_score": best_behavior_avg["f1_score"],
    },
    {
        "method": "MSNet + Behavioral Fusion",
        "feature_set": best_fusion_row["feature_set"],
        "model": best_fusion_row["model"],
        "alpha_msnet": best_fusion_row["alpha_mean"],
        "behavior_weight": best_fusion_row["behavior_weight_mean"],
        "acc": best_fusion_avg["acc"],
        "sensitivity": best_fusion_avg["sensitivity"],
        "specificity": best_fusion_avg["specificity"],
        "auc": best_fusion_avg["auc"],
        "precision": best_fusion_avg["precision"],
        "f1_score": best_fusion_avg["f1_score"],
    }
]

final_comparison = pd.DataFrame(comparison_rows)

baseline_acc = final_comparison.loc[final_comparison["method"] == "MSNet baseline", "acc"].iloc[0]
baseline_auc = final_comparison.loc[final_comparison["method"] == "MSNet baseline", "auc"].iloc[0]
baseline_f1 = final_comparison.loc[final_comparison["method"] == "MSNet baseline", "f1_score"].iloc[0]
baseline_sen = final_comparison.loc[final_comparison["method"] == "MSNet baseline", "sensitivity"].iloc[0]
baseline_spe = final_comparison.loc[final_comparison["method"] == "MSNet baseline", "specificity"].iloc[0]

final_comparison["delta_acc_vs_baseline"] = final_comparison["acc"] - baseline_acc
final_comparison["delta_auc_vs_baseline"] = final_comparison["auc"] - baseline_auc
final_comparison["delta_f1_vs_baseline"] = final_comparison["f1_score"] - baseline_f1
final_comparison["delta_sensitivity_vs_baseline"] = final_comparison["sensitivity"] - baseline_sen
final_comparison["delta_specificity_vs_baseline"] = final_comparison["specificity"] - baseline_spe

display_cols = [
    "method", "feature_set", "model", "alpha_msnet", "behavior_weight",
    "acc", "sensitivity", "specificity", "auc", "precision", "f1_score",
    "delta_acc_vs_baseline", "delta_auc_vs_baseline", "delta_f1_vs_baseline",
    "delta_sensitivity_vs_baseline", "delta_specificity_vs_baseline"
]

final_comparison = final_comparison[display_cols]

final_path = f"{OUT_DIR}/final_comparison_metrics.csv"
final_comparison.to_csv(final_path, index=False)

top_fusion = fusion_metrics_df.head(10).copy()
top_fusion_path = f"{OUT_DIR}/top_fusion_results.csv"
top_fusion.to_csv(top_fusion_path, index=False)

print("Final comparison table:")
display(final_comparison)

print("\nTop 10 fusion results:")
display(top_fusion)

print("\nSaved final comparison to:", final_path)
print("Saved top fusion results to:", top_fusion_path)

print("\nBest final method:")
best_final = final_comparison.sort_values("auc", ascending=False).iloc[0]
display(best_final.to_frame().T)

In [ ]:
import os
import pandas as pd
import numpy as np

OUT_DIR = str(PROJECT_ROOT / "outputs" / "improve_behavioral")

final_path = f"{OUT_DIR}/final_comparison_metrics.csv"
assert os.path.exists(final_path), "Missing final_comparison_metrics.csv. Run the final comparison section first."

final_comparison = pd.read_csv(final_path)

rounded = final_comparison.copy()

num_cols = [
    "alpha_msnet", "behavior_weight",
    "acc", "sensitivity", "specificity", "auc", "precision", "f1_score",
    "delta_acc_vs_baseline", "delta_auc_vs_baseline", "delta_f1_vs_baseline",
    "delta_sensitivity_vs_baseline", "delta_specificity_vs_baseline"
]

for c in num_cols:
    if c in rounded.columns:
        rounded[c] = rounded[c].astype(float).round(6)

compact_cols = [
    "method",
    "feature_set",
    "model",
    "alpha_msnet",
    "behavior_weight",
    "acc",
    "sensitivity",
    "specificity",
    "auc",
    "precision",
    "f1_score",
    "delta_acc_vs_baseline",
    "delta_auc_vs_baseline",
    "delta_f1_vs_baseline"
]

rounded_compact = rounded[compact_cols]

rounded_path = f"{OUT_DIR}/final_comparison_metrics_rounded.csv"
rounded_compact.to_csv(rounded_path, index=False)

print("Compact final comparison:")
display(rounded_compact)

print("\nSaved rounded final comparison to:", rounded_path)

best_row = rounded_compact.sort_values("auc", ascending=False).iloc[0]

print("\nBest method by AUC:")
for col in compact_cols:
    print(f"{col}: {best_row[col]}")

baseline = rounded_compact[rounded_compact["method"] == "MSNet baseline"].iloc[0]
best = best_row

print("\nImprovement over MSNet baseline:")
print("Delta Acc:", round(best["acc"] - baseline["acc"], 6))
print("Delta AUC:", round(best["auc"] - baseline["auc"], 6))
print("Delta F1 :", round(best["f1_score"] - baseline["f1_score"], 6))